# Rol 6. Responsable de explicabilidad (Chet)

**Task 1.6.** En este notebook se interpretan las pruebas de explicabilidad del modelo y se decide qué explicaciones se pueden entregar con confianza. Al final se redacta la sección 6 del memo.

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`. Los datos del enunciado están en `src/case_data.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import case_data as cd

## 1. Resultados de las pruebas

El error de eficiencia de SHAP es la diferencia máxima entre la suma de los valores SHAP y la diferencia entre la predicción y el valor base. El índice de Jaccard mide cuánto coinciden las 5 variables principales de LIME entre corridas (1 significa coincidencia total).

In [2]:
tests = cd.explainability_tests()
tests.assign(valor=tests["valor"].map("{:g}".format))

,prueba,metrica,valor
0,"SHAP, 100 secuencias de fondo",Error de eficiencia máximo (log-odds),3e-07
1,"LIME, 500 muestras","Jaccard promedio top-5, 20 corridas",0.45
2,"LIME, 5,000 muestras","Jaccard promedio top-5, 20 corridas",0.78
3,Atención vs SHAP,Correlación de rangos,0.35


In [3]:
lime = tests[tests["prueba"].str.startswith("LIME")]["valor"].to_list()
print(f"Mejora de estabilidad de LIME al pasar de 500 a 5,000 muestras: {lime[0]} -> {lime[1]} (+{lime[1] - lime[0]:.2f})")

Mejora de estabilidad de LIME al pasar de 500 a 5,000 muestras: 0.45 -> 0.78 (+0.33)


### a. Interpretación de cada resultado

| Prueba | Interpretación | ¿Se entrega? |
|---|---|---|
| SHAP (error 3e-07) | Los valores SHAP suman casi exactamente la diferencia entre la predicción y el valor base. Las atribuciones son consistentes con el modelo. | **Sí**, como explicación principal, verificando que las 100 secuencias de fondo representen a los remitentes normales. |
| LIME, 500 muestras (0.45) | Menos de la mitad de las 5 variables principales coincide entre corridas. La explicación cambia cada vez que se calcula. | **No.** |
| LIME, 5,000 muestras (0.78) | La estabilidad mejora 0.33, pero todavía varía entre corridas. | Solo como apoyo interno, nunca como explicación oficial. |
| Atención vs SHAP (0.35) | Los pesos de atención coinciden poco con las atribuciones SHAP. La atención indica dónde mira el modelo, no qué causa el puntaje. | **No.** |

### b. Explicación por audiencia

| Audiencia | Explicación | Motivo |
|---|---|---|
| Oficial de cumplimiento | Valores SHAP de cada alerta, con las variables y transacciones que más elevaron el puntaje. | Necesita revisar la alerta y redactar la narrativa con hechos concretos. |
| Regulador | Metodología documentada (modelo, datos, umbral), explicación SHAP por alerta y evidencia de su confiabilidad (error de eficiencia de 3e-07 y pruebas de estabilidad). | La ley exige explicar por qué se generó cada alerta y demostrar que la explicación es confiable. |
| Cliente señalado | Ninguna explicación del modelo. Solo la comunicación general que permita la normativa. | La regulación contra el lavado de dinero prohíbe alertar al cliente sobre un reporte, y la política de cumplimiento prohíbe indicarle qué cambiar. |

### c. Política de cumplimiento y validación adicional

Una explicación del tipo "si sus envíos fueran menores a cierto monto no habría alerta" es una **explicación contrafactual**. Entregada a un cliente señalado, funciona como una guía para evadir el sistema, por ejemplo fraccionando montos por debajo del umbral. Además, avisar al cliente pone en riesgo la investigación. Por eso la política tiene sentido.

Antes del lanzamiento se exigiría lo siguiente.

1. **Estabilidad de SHAP frente al fondo.** Repetir SHAP con varios conjuntos distintos de 100 secuencias de fondo y verificar que las variables principales se mantienen.
2. **Prueba de aleatorización del modelo.** Verificar que las explicaciones cambian al aleatorizar los pesos, para confirmar que dependen del modelo y no solo de los datos.
3. **Revisión con expertos.** Que oficiales de cumplimiento revisen las explicaciones SHAP de casos confirmados y confirmen que coinciden con tipologías conocidas.

### d. Sección 6 del memo

> **6. Explicabilidad.** Se recomienda entregar explicaciones SHAP, cuyo error de eficiencia máximo es de 3e-07 log-odds, al oficial de cumplimiento y al regulador. No se deben entregar LIME con 500 muestras, porque solo coincide en 0.45 entre corridas, ni los pesos de atención, que se correlacionan 0.35 con SHAP. Al cliente señalado no se le entrega ninguna explicación del modelo, para no enseñarle a evadir el sistema. Antes del lanzamiento se debe verificar la estabilidad de SHAP con distintos conjuntos de fondo.